# Olist Marketplace Analysis
## What Really Drives Customer Satisfaction on Brazil\'s Largest Marketplace Integrator?
**Data Analytics Hackathon — Gradient Learnings | September 2026**  
**Team Name:** GenWin  
**Team Leader:** Swati Dubey  
**Team Member:** Nitanshu Tak  

---

### Project Overview & Architecture
This notebook conducts an end-to-end empirical investigation of ~2 years of Brazilian e-commerce operations from Olist (September 2016 – October 2018). We integrate 9 relational tables into an order-grain analytical master dataset (96,478 delivered orders) to isolate the structural root causes of customer dissatisfaction.

`
       orders (99,441)
              │
  ┌───────────┼───────────────┬─────────────────┐
  ▼           ▼               ▼                 ▼
items      payments        reviews          customers
(112,650)  (103,886)      (100,000)          (99,441)
  │           │               │                 │
  └───────────┴───────────────┴─────────────────┘
                      │
                      ▼
        Master Analytical Order Grain
              (96,478 Rows)
                      │
     ┌────────────────┴────────────────┐
     ▼                                 ▼
Econometric Modeling              NLP Review Text
(sm.Logit, AUC=0.702)         (Disproportionate Ratios)
`

**Disclosure:** AI tools were utilized during this project for exploratory scaffolding, syntax debugging, and narrative structuring while the analytical design, hypothesis testing, econometric specification, and strategic interpretations were independently directed and verified.


In [ ]:
# 1. Environment Configuration & Libraries
!pip install -q statsmodels scikit-learn

import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
PALETTE = ['#2E4374', '#4C7DAA', '#7FB7C4', '#F2A65A', '#E15759', '#8CC084']
sns.set_palette(PALETTE)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 12
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [ ]:
# 2. Resilient Data Ingestion Pipeline
DATA_DIR = 'data'
SHEETS = [
    'orders', 'order_items', 'order_payments', 'order_reviews',
    'customers', 'products', 'sellers', 'geolocation', 'category_translation'
]

def try_load_from_folder(folder):
    if not os.path.isdir(folder):
        return None
    dfs = {}
    for s in SHEETS:
        path = os.path.join(folder, f'{s}.csv')
        if not os.path.exists(path):
            return None
        dfs[s] = pd.read_csv(path)
    return dfs

dfs = try_load_from_folder(DATA_DIR)
if dfs is None:
    try:
        from google.colab import files
        print('No local data/ directory found. Please upload the workbook or CSV package.')
        uploaded = files.upload()
        xlsx_path = list(uploaded.keys())[0]
        xl = pd.ExcelFile(xlsx_path)
        dfs = {s: xl.parse(s) for s in SHEETS}
    except Exception as e:
        print('Data loading initialized via current directory fallback or placeholder.')

if dfs is not None:
    orders = dfs['orders']
    items = dfs['order_items']
    pay = dfs['order_payments']
    rev = dfs['order_reviews']
    cust = dfs['customers']
    prod = dfs['products']
    sell = dfs['sellers']
    geo = dfs['geolocation']
    cat = dfs['category_translation']
    for name, d in dfs.items():
        print(f'{name:24s}: {d.shape}')


In [ ]:
# 3. Relational Harmonization & Feature Engineering
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

rev['review_creation_date'] = pd.to_datetime(rev['review_creation_date'])
rev['review_answer_timestamp'] = pd.to_datetime(rev['review_answer_timestamp'])
rev = rev.drop_duplicates(subset='review_id', keep='first')

prod['product_category_name'] = prod['product_category_name'].fillna('unknown')
prod = prod.merge(cat, on='product_category_name', how='left')
prod['product_category_name_english'] = prod['product_category_name_english'].fillna('unknown')

geo_agg = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

order_items_agg = items.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    item_price_total=('price', 'sum'),
    freight_total=('freight_value', 'sum'),
    n_sellers=('seller_id', 'nunique'),
).reset_index()

pay_primary = pay.sort_values('payment_value', ascending=False).drop_duplicates('order_id')[['order_id', 'payment_type']]
pay_agg = pay.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    max_installments=('payment_installments', 'max'),
    n_payments=('payment_sequential', 'count'),
).reset_index().merge(pay_primary, on='order_id', how='left')

rev_order = rev.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')[
    ['order_id', 'review_score', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
]

df = orders.merge(cust, on='customer_id', how='left')
df = df.merge(order_items_agg, on='order_id', how='left')
df = df.merge(pay_agg, on='order_id', how='left')
df = df.merge(rev_order, on='order_id', how='left')

df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delay_vs_estimate'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['is_late'] = df['delay_vs_estimate'] > 0
df['purchase_yearmonth'] = df['order_purchase_timestamp'].dt.to_period('M')

delivered = df[df['order_status'] == 'delivered'].copy()

item_full = items.merge(
    prod[['product_id', 'product_category_name_english', 'product_weight_g',
          'product_length_cm', 'product_height_cm', 'product_width_cm']],
    on='product_id', how='left'
)
item_full = item_full.merge(sell, on='seller_id', how='left')
item_full = item_full.merge(orders[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id', how='left')
item_full = item_full.merge(rev_order[['order_id', 'review_score']], on='order_id', how='left')

print(f'Master analytical table grain: {df.shape}')
print(f'Delivered orders cohort: {delivered.shape}')
print(f'Item-level analytical table: {item_full.shape}')


In [ ]:
# 4. Delivery Performance vs. Review Score (The Threshold Cliff)
d2 = df[(df['order_status'] == 'delivered') & df['delay_vs_estimate'].notna() & df['review_score'].notna()].copy()

def bucket(x):
    if x <= -7: return 'Very early (7+ days)'
    if x <= -1: return 'Early (1-6 days)'
    if x == 0: return 'On time'
    if x <= 7: return 'Late (1-7 days)'
    return 'Very late (7+ days)'

d2['delay_bucket'] = d2['delay_vs_estimate'].apply(bucket)
bucket_order = ['Very early (7+ days)', 'Early (1-6 days)', 'On time', 'Late (1-7 days)', 'Very late (7+ days)']
summary = d2.groupby('delay_bucket')['review_score'].agg(['mean', 'count']).reindex(bucket_order)
print(summary)

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ['#2E4374', '#4C7DAA', '#7FB7C4', '#F2A65A', '#E15759']
bars = ax.bar(summary.index, summary['mean'], color=colors, edgecolor='#333333', width=0.55)
for b, cnt in zip(bars, summary['count']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1, f'n={cnt:,}', ha='center', fontsize=9, color='#333333')
ax.set_ylim(0, 5.2)
ax.set_ylabel('Average Review Score (1-5 Stars)')
ax.set_title('Threshold Effect: Rating Collapses Immediately Past Promised Date')
plt.tight_layout()
plt.show()

corr, p = stats.pearsonr(d2['delay_vs_estimate'], d2['review_score'])
print(f'Pearson correlation across raw days: r = {corr:.3f} (p = {p:.1e})')


In [ ]:
# 5. Multivariate Econometric Modeling (sm.Logit)
d6 = df[(df['order_status'] == 'delivered') & df['review_score'].notna() & df['delay_vs_estimate'].notna()].copy()
d6['low_review'] = (d6['review_score'] <= 2).astype(int)

d6['delay_days_capped'] = d6['delay_vs_estimate'].clip(-30, 30)
d6['n_items_capped'] = d6['n_items'].clip(1, 10)
d6['freight_ratio'] = (d6['freight_total'] / d6['item_price_total'].replace(0, np.nan)).clip(0, 2)
d6['is_multi_seller'] = (d6['n_sellers'] > 1).astype(int)
d6['log_price'] = np.log1p(d6['item_price_total'])

feat_cols = ['delay_days_capped', 'n_items_capped', 'freight_ratio', 'is_multi_seller', 'log_price', 'max_installments']
model_df = d6.dropna(subset=feat_cols + ['low_review'])

X = sm.add_constant(model_df[feat_cols])
y = model_df['low_review']
logit = sm.Logit(y, X).fit(disp=0)
print(logit.summary())

params = logit.params
conf = logit.conf_int()
conf.columns = ['2.5%', '97.5%']
conf['OR'] = np.exp(params)
conf['OR_low'] = np.exp(conf['2.5%'])
conf['OR_high'] = np.exp(conf['97.5%'])
print('\nEstimated Odds Ratios (95% CI):')
print(conf[['OR', 'OR_low', 'OR_high']])


In [ ]:
# 6. Econometric Stress Testing & Confound Isolation
# Check 1: Multicollinearity (VIF)
X_vif = sm.add_constant(model_df[feat_cols])
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('Variance Inflation Factors (Threshold < 5.0):')
print(vif_data)

# Check 2: Out-of-Sample Holdout (75/25 Split)
train, test = train_test_split(model_df, test_size=0.25, random_state=42, stratify=model_df['low_review'])
Xtr, Xte = sm.add_constant(train[feat_cols]), sm.add_constant(test[feat_cols])
logit_tr = sm.Logit(train['low_review'], Xtr).fit(disp=0)
auc = roc_auc_score(test['low_review'], logit_tr.predict(Xte))
print(f'\nOut-of-Sample Test AUC: {auc:.3f}')
print('Out-of-Sample Odds Ratios:')
print(np.exp(logit_tr.params).round(2))

# Check 3: Confound Isolation Test
ontime_small = d6[(d6['delay_vs_estimate'] <= 0) & (d6['n_items'] <= 3)]
confound_check = ontime_small.groupby('is_multi_seller')['low_review'].agg(['mean', 'count'])
confound_check.index = ['Single Seller', 'Multiple Sellers']
print('\nConfound Isolation: Low-review rate on strictly on-time, small-basket orders:')
print(confound_check)


In [ ]:
# 7. Qualitative NLP Review Intelligence
rev_text = rev.dropna(subset=['review_comment_message'])
stopwords = set(a o os as um uma umas uns de da do das dos em no na nos nas por para com que se e é foi ser tem
tinha muito mais mas ao aos à às pelo pela pelos pelas isso este esta estes estas esse essa desde até como já
não nao sim minha meu meus minhas ele ela eles elas eu você voce nós nos me mim te lhe também bem so só ainda
depois antes quando onde qual quais foi era são vou vai fazer fiz ter há havia produto produtos comprei compra
recebi entrega entregue.split())

def tokenize(t):
    t = re.sub(r'[^a-zà-ÿ\s]', ' ', t.lower())
    return [w for w in t.split() if len(w) > 3 and w not in stopwords]

low_txt = rev_text[rev_text['review_score'] <= 2]['review_comment_message']
high_txt = rev_text[rev_text['review_score'] >= 4]['review_comment_message']

low_words, high_words = Counter(), Counter()
for t in low_txt: low_words.update(tokenize(t))
for t in high_txt: high_words.update(tokenize(t))

low_total, high_total = sum(low_words.values()), sum(high_words.values())
token_rows = []
for w, c in low_words.items():
    if c < 30: continue
    ratio = (c / low_total) / (high_words.get(w, 0.5) / high_total)
    token_rows.append((w, c, high_words.get(w, 0), round(ratio, 1)))

top_words = pd.DataFrame(
    sorted(token_rows, key=lambda x: -x[3])[:15],
    columns=['Portuguese Term', 'Count in Bad Reviews', 'Count in High Reviews', 'Disproportionate Ratio']
)
print('Top Disproportionate Sentiment Terms in 1-2 Star Reviews:')
print(top_words)
